# Teacher Foundation Model — Training Pipeline

Baseline: `v1.0-baseline`  
Dataset: Google Drive → `MarketFoundation/storage/`

## Cell 1 — GPU Information

In [ ]:
import torch
import platform
import os

print("=" * 60)
print("Python:", platform.python_version())
print("Torch :", torch.__version__)
print("CUDA  :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU   :", torch.cuda.get_device_name(0))
    print("VRAM  :", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

print("=" * 60)

## Cell 2 — Clone Repository & Checkout Baseline Tag

In [ ]:
!git clone https://github.com/sandeep999-cyber/emptyu.git
%cd emptyu
!git checkout v1.0-baseline

## Cell 3 — Install Dependencies

In [ ]:
!pip install -r requirements.txt

## Cell 4 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Cell 5 — Configure Dataset Paths

In [ ]:
import os
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/MarketFoundation")

# Source directories on Drive
DATA_ROOT = DRIVE_ROOT / "storage"
CHECKPOINT_ROOT = DRIVE_ROOT / "checkpoints"
LOG_ROOT = DRIVE_ROOT / "logs"
EVAL_ROOT = DRIVE_ROOT / "evaluation"

# Create Drive directories (if first run)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
LOG_ROOT.mkdir(parents=True, exist_ok=True)
EVAL_ROOT.mkdir(parents=True, exist_ok=True)

# Symlink repo paths → Drive paths so config works without modification
if not Path("storage").exists():
    !ln -sf "{DATA_ROOT}" storage
if not Path("models").exists():
    !ln -sf "{CHECKPOINT_ROOT}" models
if not Path("logs").exists():
    !ln -sf "{LOG_ROOT}" logs
if not Path("evaluation").exists():
    !ln -sf "{EVAL_ROOT}" evaluation

print(f"Dataset : {DATA_ROOT}")
print(f"Logs    : {LOG_ROOT}")
print(f"Models  : {CHECKPOINT_ROOT}")
print(f"Eval    : {EVAL_ROOT}")

## Cell 6 — Verify Dataset

In [ ]:
from pathlib import Path
import json

manifest = Path("storage/training/training_manifest_v1.json")
fingerprint = Path("storage/training/dataset_fingerprint.json")

print(f"Manifest exists   : {manifest.exists()}")
print(f"Fingerprint exists: {fingerprint.exists()}")

if fingerprint.exists():
    fp = json.loads(fingerprint.read_text())
    print("Fingerprint:", json.dumps(fp, indent=2))

## Cell 7 — Verify Snapshot

In [ ]:
from pathlib import Path

snapshot_dir = Path("storage/training/snapshots")
print(f"Snapshot dir: {snapshot_dir}")
print(f"Exists: {snapshot_dir.exists()}")

if snapshot_dir.exists():
    for p in sorted(snapshot_dir.iterdir()):
        print(f"  {p.name}")

## Cell 8 — Smoke Test

In [ ]:
!python -m src.training.train_teacher \
    --model-config configs/model_v1.yaml \
    --optimizer-config configs/optimizer_v1.yaml \
    --trainer-config configs/trainer_v1.yaml \
    --smoke

## Cell 9 — Pilot Training

In [ ]:
!python -m src.training.train_teacher \
    --model-config configs/model_v1.yaml \
    --optimizer-config configs/optimizer_v1.yaml \
    --trainer-config configs/trainer_v1.yaml

## Cell 10 — Resume Training (if needed)

In [ ]:
# Uncomment and fill in your run ID:
# !python -m src.training.train_teacher \
#     --model-config configs/model_v1.yaml \
#     --optimizer-config configs/optimizer_v1.yaml \
#     --trainer-config configs/trainer_v1.yaml \
#     --resume models/foundation/teacher_v1/<RUN_ID>

## Cell 11 — Find Latest Checkpoint

In [ ]:
from pathlib import Path

checkpoint_dirs = sorted(Path("models/foundation/teacher_v1").iterdir()) if Path("models/foundation/teacher_v1").exists() else []

if checkpoint_dirs:
    latest = checkpoint_dirs[-1]
    print(f"Latest checkpoint dir: {latest}")
    %env CHECKPOINT_DIR {latest}
else:
    print("No checkpoints found. Run training first.")

## Cell 12 — Clustering Evaluation

In [ ]:
!python -m src.evaluation.embedding.clustering \
    --checkpoint "$CHECKPOINT_DIR" \
    --split train \
    --pooling mean

## Cell 13 — Retrieval Evaluation

In [ ]:
!python -m src.evaluation.embedding.retrieval \
    --checkpoint "$CHECKPOINT_DIR" \
    --split validation \
    --pooling mean

## Cell 14 — Temporal Consistency

In [ ]:
!python -m src.evaluation.embedding.temporal_consistency \
    --checkpoint "$CHECKPOINT_DIR" \
    --split validation \
    --pooling mean

## Cell 15 — Linear Probe

In [ ]:
!python -m src.evaluation.embedding.linear_probe \
    --checkpoint "$CHECKPOINT_DIR" \
    --pooling mean

## Cell 16 — Visualization

In [ ]:
!python -m src.evaluation.embedding.visualization \
    --checkpoint "$CHECKPOINT_DIR" \
    --method pca \
    --pooling mean

## Cell 17 — Archive Results

In [ ]:
import shutil

archive_path = "/content/drive/MyDrive/MarketFoundation/phase2_results"
shutil.make_archive(archive_path, "zip", "evaluation")
print(f"Results archived to {archive_path}.zip")

## Optional — Save Environment Snapshot

In [ ]:
!pip freeze > environment.txt
!nvidia-smi